In [9]:
%pip install --upgrade pip setuptools wheel
%pip install undetected-chromedriver

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import re
import time
import numpy as np
import undetected_chromedriver as uc 

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select

import requests
from bs4 import BeautifulSoup
import altair as alt

In [11]:
options = uc.ChromeOptions()
driver = uc.Chrome(options=options, version_main=150)

In [12]:
driver.get("https://www.officialgazette.gov.ph/past-sona-speeches/")

In [13]:
df=pd.read_csv('aquino_english.csv')
df

,president,date,title,link,venue
0,Benigno S. Aquino III,26-Jul-10,State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City"
1,Benigno S. Aquino III,25-Jul-11,Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City"
2,Benigno S. Aquino III,23-Jul-12,Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City"
3,Benigno S. Aquino III,22-Jul-13,Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City"
4,Benigno S. Aquino III,28-Jul-14,Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City"
5,Benigno S. Aquino III,27-Jul-15,Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City"


In [14]:
dataset = df.to_dict(orient='records')
dataset

[{'president': 'Benigno S. Aquino III',
  'date': '26-Jul-10',
  'title': 'State of the Nation Address',
  'link': 'https://www.officialgazette.gov.ph/2010/07/26/state-of-the-nation-address-2010-en/',
  'venue': 'Batasang Pambansa, Quezon City'},
 {'president': 'Benigno S. Aquino III',
  'date': '25-Jul-11',
  'title': 'Second State of the Nation Address',
  'link': 'https://www.officialgazette.gov.ph/2011/07/25/benigno-s-aquino-iii-second-state-of-the-nation-address-july-25-2011-en/',
  'venue': 'Batasang Pambansa, Quezon City'},
 {'president': 'Benigno S. Aquino III',
  'date': '23-Jul-12',
  'title': 'Third State of the Nation Address',
  'link': 'https://www.officialgazette.gov.ph/2012/07/23/english-translation-benigno-s-aquino-iii-third-state-of-the-nation-address-july-23-2012/',
  'venue': 'Batasang Pambansa, Quezon City'},
 {'president': 'Benigno S. Aquino III',
  'date': '22-Jul-13',
  'title': 'Fourth State of the Nation Address',
  'link': 'https://www.officialgazette.gov.ph/

In [15]:
total_speeches = len(dataset)

for idx, data in enumerate(dataset):
    href = data['link']
    print(f"[{idx + 1}/{total_speeches}] Extracting text from: {href}")
    
    try:
        # Navigate directly in the primary active browser session
        driver.get(href)
        time.sleep(4)  # Give anti-bot engine room to breathe
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        speech_content = ""
        
        # Strategy A: Target Gazette's standard content container
        elements = soup.find_all(class_='large-9 large-centered columns')
        if len(elements) > 1:
            speech_content = elements[1].text.strip()
        elif len(elements) == 1:
            speech_content = elements[1].text.strip()
            
        # Strategy B: Fallback layout structural tags
        if not speech_content or len(speech_content) < 200:
            article = soup.find('article') or soup.find(class_='entry-content') or soup.find(id='content')
            if article:
                speech_content = article.text.strip()
                
        # Strategy C: Absolute text dump safety net
        if not speech_content or len(speech_content) < 200:
            speech_content = soup.body.text.strip() if soup.body else "[EMPTY BODY]"

        data['speech_text'] = speech_content
        
    except Exception as e:
        print(f"   ⚠️ Lost page connection or error: {e}")
        data['speech_text'] = f"[ERROR RETRIEVING TEXT: {str(e)}]"

print("\n🎉 Complete! All rows updated.")

[1/6] Extracting text from: https://www.officialgazette.gov.ph/2010/07/26/state-of-the-nation-address-2010-en/
[2/6] Extracting text from: https://www.officialgazette.gov.ph/2011/07/25/benigno-s-aquino-iii-second-state-of-the-nation-address-july-25-2011-en/
[3/6] Extracting text from: https://www.officialgazette.gov.ph/2012/07/23/english-translation-benigno-s-aquino-iii-third-state-of-the-nation-address-july-23-2012/
[4/6] Extracting text from: https://www.officialgazette.gov.ph/2013/07/22/english-benigno-s-aquino-iii-fourth-state-of-the-nation-address-july-22-2013/
[5/6] Extracting text from: https://www.officialgazette.gov.ph/2014/07/28/english-benigno-s-aquino-iii-fifth-state-of-the-nation-address-july-28-2014/
[6/6] Extracting text from: https://www.officialgazette.gov.ph/2015/07/27/english-president-aquino-sixth-sona/

🎉 Complete! All rows updated.


In [16]:
# 1. Convert your complete dataset list into a DataFrame
df_final = pd.DataFrame(dataset)
df_final

,president,date,title,link,venue,speech_text
0,Benigno S. Aquino III,26-Jul-10,State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City",State of the Nation Address of His Excellency\...
1,Benigno S. Aquino III,25-Jul-11,Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof \nHis Excellen...
2,Benigno S. Aquino III,23-Jul-12,Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof\nHis Excellenc...
3,Benigno S. Aquino III,22-Jul-13,Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof\nHis Excellenc...
4,Benigno S. Aquino III,28-Jul-14,Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof\nHis Excellenc...
5,Benigno S. Aquino III,27-Jul-15,Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City",The 2015 State of the Nation Address\n[Basahin...


In [17]:
df_final.to_csv('aquino_english_scraped.csv', index=False)